In [ ]:
import numpy as np
import pandas as pd
import scanpy as sc
from scipy.sparse import csr_matrix
from pathlib import Path
import os

In [ ]:
PATH_TO_DATA = Path("../../../../../sandbox/parse_pbmc/")
adata = sc.read_h5ad(PATH_TO_DATA / "Parse_10M_PBMC_cytokines.h5ad")  

In [ ]:
sc.pp.normalize_total(adata, target_sum=1e4)
sc.pp.log1p(adata)

hvg_path = Path("../../data_dir/perturbdiff_hvg/pbmc_real_selected_genes.txt")
hvg_all = hvg_path.read_text().splitlines()
hvg = [g for g in hvg_all if g in adata.var_names]
print(f"Selected {len(hvg)} / {len(hvg_all)} PerturbDiff HVGs present in adata")
adata = adata[:, hvg]

adata.write_h5ad(Path("../../data_dir/pbmc_parse.h5ad"))

## When the subset to HVG is completed 

In [ ]:
PATH_TO_DATA = Path("../../data_dir/")
# Re-run cells 1–2 if Parse_10M_PBMC_cytokines.h5ad changed (writes pbmc_parse.h5ad).
adata = sc.read_h5ad(PATH_TO_DATA / "pbmc_parse.h5ad")  

## Cytokine labels (STATE vs Parse)

Parse `obs.cytokine` uses alternate names for 3 conditions; `sample` suffix matches STATE. Remap before splits so holdout perturbations (e.g. `TGF-beta1`) land in the right `split3` bucket.

In [ ]:
# Parse obs.cytokine uses receptor/product names for 3 conditions; sample suffix matches STATE.
CYTOKINE_MAP = {
    "LAP-TGF-beta1": "TGF-beta1",  # latent TGF-β1 — same treatment as STATE TGF-beta1
    "Megalin": "C3a",              # Parse labels C3a wells as Megalin (LRP2)
    "IL-18Ra": "IL-18",            # IL-18 treatment; obs names the receptor
}
parse_cytokine = adata.obs["cytokine"].astype(str).copy()
adata.obs["cytokine"] = parse_cytokine.replace(CYTOKINE_MAP)

if "sample" in adata.obs.columns:
    sample_cytokine = adata.obs["sample"].astype(str).str.split("_").str[1]
    mismatches = adata.obs["cytokine"] != sample_cytokine
    if mismatches.any():
        summary = (
            pd.DataFrame({
                "parse_cytokine": parse_cytokine[mismatches],
                "remapped": adata.obs.loc[mismatches, "cytokine"],
                "sample_suffix": sample_cytokine[mismatches],
            })
            .value_counts()
            .reset_index(name="n_cells")
        )
        raise ValueError(f"cytokine/sample mismatch after remap:\n{summary}")
    print("cytokine labels match sample suffix for all cells")
elif (parse_cytokine != adata.obs["cytokine"]).any():
    print("Remapped cytokines:", CYTOKINE_MAP)

In [ ]:
adata.obs["condition"] = adata.obs.apply(lambda x: x["donor"] + "_" + x["cytokine"], axis=1)
adata.obs["is_control"] = adata.obs.apply(lambda x: True if x["cytokine"]=="PBS" else False, axis=1)

Seeded subsampling of the anndata

In [ ]:
# adata_subsample = sc.pp.subsample(adata, fraction=0.05, random_state=0, copy=True)
# sc.tl.pca(adata_subsample)
# sc.pp.neighbors(adata_subsample)
# sc.tl.umap(adata_subsample)

In [ ]:
# sc.pl.umap(adata_subsample, color=["donor", "cell_type"], s=10)

In [ ]:
# for donor in adata_subsample.obs.donor.unique():
#     sc.pl.umap(adata_subsample, color=["donor"], s=10, groups=[donor])

In [ ]:
# for cytokine in adata_subsample.obs.cytokine.unique():
#     sc.pl.umap(adata_subsample, color=["cytokine"], s=10, groups=[cytokine])

In [ ]:
# sc.pl.umap(adata[adata_subsample.obs.donor!="Donor8"], color=["donor"], s=10)

In [ ]:
# sc.pl.umap(adata_subsample[adata_subsample.obs.donor!="Donor8"], color=["cytokine"], groups=["IL-7"], s=10)
# sc.pl.umap(adata_subsample[adata_subsample.obs.donor!="Donor8"], color=["cytokine"], groups=["PBS"], s=10)

## Splits

We define three splitting schemes (stored in `obs` as `split1`, `split2`, `split3`):

* **Split 1:** Pretrain on 11 donors; finetune on **Donor2** with all cytokines (in-distribution, no perturbation holdout).
* **Split 2:** Same as split 1, but finetune on **Donor8** instead.
* **Split 3:** STATE preprint few-shot split — finetune donors **1, 4, 9, 12** are partitioned by cytokine into `finetune` / `val` / `test`; donors 2, 3, 5, 6, 7, 8, 10, 11 stay in `train`.

## Split 1: fine-tune on all of Donor2, no holdout

In [ ]:
adata.obs["split1"] = "finetune"
adata.obs.loc[adata.obs.donor!="Donor2", "split1"] = "train"

# adata_subsample.obs["split1"] = "finetune"
# adata_subsample.obs.loc[adata_subsample.obs.donor!="Donor2", "split1"] = "train"

In [ ]:
print(adata.obs["split1"].value_counts())
# print(adata_subsample.obs["split1"].value_counts())

In [ ]:
print(pd.crosstab(adata.obs.split1, adata.obs.donor))
# print(pd.crosstab(adata_subsample.obs.split1, adata_subsample.obs.donor))

## Split 2: fine-tune on all of Donor8, no holdout

In [ ]:
adata.obs["split2"] = "finetune"
adata.obs.loc[adata.obs.donor!="Donor8", "split2"] = "train"

# adata_subsample.obs["split2"] = "finetune"
# adata_subsample.obs.loc[adata_subsample.obs.donor!="Donor8", "split2"] = "train"

In [ ]:
print(adata.obs["split2"].value_counts())
# print(adata_subsample.obs["split2"].value_counts())

In [ ]:
print(pd.crosstab(adata.obs.split2, adata.obs.donor))
# print(pd.crosstab(adata_subsample.obs.split2, adata_subsample.obs.donor))

## Split 3: STATE preprint donor split (Donors 1, 4, 9, 12)

For each finetune donor, cytokines are assigned to `val` (4 cytokines), `test` (62 cytokines), or `finetune` (the remaining 25). All other donors remain entirely in `train`.

In [ ]:
config = {
    "Donor1": {
        "val": ['G-CSF', 'IFN-beta', 'M-CSF', 'IGF-1'],
        "test": ['C5a', 'TWEAK', 'LIF', 'IL-17C', 'FGF-beta', 'BAFF', 'FLT3L', 'IFN-lambda2', '4-1BBL', 'IL-17E', 'IL-2', 'IL-19', 'PRL', 'IL-16', 'IL-7', 'GM-CSF', 'IL-17A', 'IL-13', 'IL-36-alpha', 'PSPN', 'EPO', 'IL-9', 'IL-31', 'IL-11', 'LT-alpha1-beta2', 'TL1A', 'IL-32-beta', 'IFN-omega', 'IL-8', 'Noggin', 'IFN-epsilon', 'IL-17F', 'Leptin', 'IL-33', 'IL-35', 'OSM', 'IL-34', 'HGF', 'IL-1Ra', 'LIGHT', 'CD27L', 'SCF', 'IL-1-alpha', 'TGF-beta1', 'CD30L', 'IL-24', 'IFN-gamma', 'IFN-alpha1', 'IFN-lambda1', 'GITRL', 'VEGF', 'ADSF', 'IL-15', 'EGF', 'IL-3', 'IL-17B', 'IL-6', 'IL-4', 'IL-10', 'CT-1', 'IFN-lambda3', 'TRAIL']
    }, 
    "Donor4": {
        "val": ['G-CSF', 'IFN-beta', 'M-CSF', 'IGF-1'],
        "test": ['C5a', 'TWEAK', 'LIF', 'IL-17C', 'FGF-beta', 'BAFF', 'FLT3L', 'IFN-lambda2', '4-1BBL', 'IL-17E', 'IL-2', 'IL-19', 'PRL', 'IL-16', 'IL-7', 'GM-CSF', 'IL-17A', 'IL-13', 'IL-36-alpha', 'PSPN', 'EPO', 'IL-9', 'IL-31', 'IL-11', 'LT-alpha1-beta2', 'TL1A', 'IL-32-beta', 'IFN-omega', 'IL-8', 'Noggin', 'IFN-epsilon', 'IL-17F', 'Leptin', 'IL-33', 'IL-35', 'OSM', 'IL-34', 'HGF', 'IL-1Ra', 'LIGHT', 'CD27L', 'SCF', 'IL-1-alpha', 'TGF-beta1', 'CD30L', 'IL-24', 'IFN-gamma', 'IFN-alpha1', 'IFN-lambda1', 'GITRL', 'VEGF', 'ADSF', 'IL-15', 'EGF', 'IL-3', 'IL-17B', 'IL-6', 'IL-4', 'IL-10', 'CT-1', 'IFN-lambda3', 'TRAIL']
    },
    "Donor9": {
        "val": ['G-CSF', 'IFN-beta', 'M-CSF', 'IGF-1'],
        "test": ['C5a', 'TWEAK', 'LIF', 'IL-17C', 'FGF-beta', 'BAFF', 'FLT3L', 'IFN-lambda2', '4-1BBL', 'IL-17E', 'IL-2', 'IL-19', 'PRL', 'IL-16', 'IL-7', 'GM-CSF', 'IL-17A', 'IL-13', 'IL-36-alpha', 'PSPN', 'EPO', 'IL-9', 'IL-31', 'IL-11', 'LT-alpha1-beta2', 'TL1A', 'IL-32-beta', 'IFN-omega', 'IL-8', 'Noggin', 'IFN-epsilon', 'IL-17F', 'Leptin', 'IL-33', 'IL-35', 'OSM', 'IL-34', 'HGF', 'IL-1Ra', 'LIGHT', 'CD27L', 'SCF', 'IL-1-alpha', 'TGF-beta1', 'CD30L', 'IL-24', 'IFN-gamma', 'IFN-alpha1', 'IFN-lambda1', 'GITRL', 'VEGF', 'ADSF', 'IL-15', 'EGF', 'IL-3', 'IL-17B', 'IL-6', 'IL-4', 'IL-10', 'CT-1', 'IFN-lambda3', 'TRAIL']
    },
    "Donor12": {
        "val": ['G-CSF', 'IFN-beta', 'M-CSF', 'IGF-1'],
        "test": ['C5a', 'TWEAK', 'LIF', 'IL-17C', 'FGF-beta', 'BAFF', 'FLT3L', 'IFN-lambda2', '4-1BBL', 'IL-17E', 'IL-2', 'IL-19', 'PRL', 'IL-16', 'IL-7', 'GM-CSF', 'IL-17A', 'IL-13', 'IL-36-alpha', 'PSPN', 'EPO', 'IL-9', 'IL-31', 'IL-11', 'LT-alpha1-beta2', 'TL1A', 'IL-32-beta', 'IFN-omega', 'IL-8', 'Noggin', 'IFN-epsilon', 'IL-17F', 'Leptin', 'IL-33', 'IL-35', 'OSM', 'IL-34', 'HGF', 'IL-1Ra', 'LIGHT', 'CD27L', 'SCF', 'IL-1-alpha', 'TGF-beta1', 'CD30L', 'IL-24', 'IFN-gamma', 'IFN-alpha1', 'IFN-lambda1', 'GITRL', 'VEGF', 'ADSF', 'IL-15', 'EGF', 'IL-3', 'IL-17B', 'IL-6', 'IL-4', 'IL-10', 'CT-1', 'IFN-lambda3', 'TRAIL']
    }
}

finetune_donor_list = ["Donor1", "Donor4", "Donor9", "Donor12"]

In [ ]:
def split3_holdout_cytokines(config):
    holdout = set()
    for donor_cfg in config.values():
        holdout.update(donor_cfg["val"])
        holdout.update(donor_cfg["test"])
    return holdout

holdout_cytokines = split3_holdout_cytokines(config)
present = set(adata.obs["cytokine"].unique())
missing = holdout_cytokines - present
assert not missing, f"Holdout cytokines missing from obs.cytokine: {sorted(missing)}"
print(f"All {len(holdout_cytokines)} split3 holdout cytokines present in obs.cytokine")

In [ ]:
adata.obs["split3"] = "train"
for donor in finetune_donor_list:
    donor_mask = adata.obs["donor"] == donor
    adata.obs.loc[donor_mask, "split3"] = "finetune"
    adata.obs.loc[donor_mask & adata.obs["cytokine"].isin(config[donor]["val"]), "split3"] = "val"
    adata.obs.loc[donor_mask & adata.obs["cytokine"].isin(config[donor]["test"]), "split3"] = "test"

print(adata.obs["split3"].value_counts())
pd.crosstab(adata.obs["split3"], adata.obs["donor"], margins=True)


In [ ]:
# adata_subsample.obs["split3"] = "train"
# for donor in finetune_donor_list:
#     donor_mask = adata_subsample.obs["donor"] == donor
#     adata_subsample.obs.loc[donor_mask, "split3"] = "finetune"
#     adata_subsample.obs.loc[donor_mask & adata_subsample.obs["cytokine"].isin(config[donor]["val"]), "split3"] = "val"
#     adata_subsample.obs.loc[donor_mask & adata_subsample.obs["cytokine"].isin(config[donor]["test"]), "split3"] = "test"

# print(adata_subsample.obs["split3"].value_counts())
# pd.crosstab(adata_subsample.obs["split3"], adata_subsample.obs["donor"], margins=True)

In [ ]:
# PCA_FIT_FRACTION = 0.15

adata_train_ft = adata[adata.obs["split3"].isin(["train", "finetune"])]
# adata_pca_fit = sc.pp.subsample(adata, fraction=PCA_FIT_FRACTION, random_state=0, copy=True)
# adata_train_ft_pca = adata_pca_fit[adata_pca_fit.obs["split3"].isin(["train", "finetune"])].copy()
# print(
#     f"PCA fit cells: {adata_train_ft_pca.n_obs:,} / {adata.n_obs:,} total "
#     f"({100 * adata_train_ft_pca.n_obs / adata.n_obs:.1f}%)"
# )

In [ ]:
print(f"Fitting sc.tl.pca on {adata_train_ft.n_obs:,} cells with n_comps=256...")
sc.tl.pca(adata_train_ft, n_comps=256)

In [ ]:
adata.varm["PCs"] = adata_train_ft.varm["PCs"]

adata.uns["means_train"] = np.asarray(adata_train_ft.X.mean(0)).ravel()

In [ ]:
X_projected = adata.X @ csr_matrix(adata.varm["PCs"])
mean_projected = adata.uns["means_train"] @ csr_matrix(adata.varm["PCs"])
adata.obsm["X_pca"] = X_projected - mean_projected

## Store PCA and STATE (SE) embeddings on the same AnnData

PCA and SE are **cell embeddings** (256-D vs ~2048-D), so they go in `adata.obsm`, not `adata.layers` (layers must match gene `X`). Training uses `--obsm-key X_pca` or `--obsm-key X_state`.

- **PCA** is already in `obsm["X_pca"]` (fit on `split3` train+finetune; val/test held out of the fit).
- **SE-600M** is copied from the encode dump into `obsm["X_state"]`. The existing dump is a `.npy` with **the same cell order** as this h5ad.

Attaching SE will add ~75 GB to the written h5ad. If the dump is missing:

```bash
SKIP_DOWNLOAD=1 BATCH_SIZE=256 \
  INPUT_H5AD=data_dir/pbmc_parse_with_splits.h5ad \
  OUTPUT_H5AD=results_dir/arc_state/se/pbmc_parse_with_splits_se.npy \
  sbatch --mem=768G --time=48:00:00 sbatch/third_party/encode_se.sbatch
```
Then re-run the attach + write cells.


In [ ]:
REPO = Path("../../")
SE_PATH = REPO / "results_dir/arc_state/se/pbmc_parse_with_splits_se.npy"
SE_KEY = "X_state"


def attach_state_embeddings(adata, se_path, *, key=SE_KEY) -> bool:
    """Copy frozen SE-600M vectors into ``adata.obsm[key]``.

    ``.npy`` dumps are assumed to share this AnnData's cell order (encode from
    the same h5ad). ``.h5ad`` dumps are aligned by ``obs_names``.
    """
    se_path = Path(se_path)
    if not se_path.exists():
        print(
            f"SE dump not found: {se_path}\n"
            "Skipping obsm['X_state']. Encode with sbatch/third_party/encode_se.sbatch "
            "then re-run this cell and the write."
        )
        return False
    if se_path.suffix == ".npy":
        z = np.load(se_path, mmap_mode="r")
        if z.shape[0] != adata.n_obs:
            raise ValueError(
                f"SE npy has {z.shape[0]} rows but adata has {adata.n_obs} cells."
            )
        adata.obsm[key] = np.asarray(z, dtype=np.float32)
        return True
    se = sc.read_h5ad(se_path)
    if key not in se.obsm:
        raise KeyError(f"obsm[{key!r}] missing in {se_path}; have {list(se.obsm)}")
    if not se.obs_names.equals(adata.obs_names):
        missing = adata.obs_names.difference(se.obs_names)
        if len(missing):
            raise ValueError(
                f"{len(missing)} cells in adata missing from SE dump, e.g. {list(missing[:5])}"
            )
        se = se[adata.obs_names].copy()
    adata.obsm[key] = np.asarray(se.obsm[key], dtype=np.float32)
    return True


if "X_pca" not in adata.obsm:
    raise RuntimeError("Run the PCA cells first (obsm['X_pca']).")

attached = attach_state_embeddings(adata, SE_PATH)
adata.uns["pca_obsm_key"] = "X_pca"
adata.uns["se_obsm_key"] = SE_KEY if attached else None
adata.uns["se_source"] = str(SE_PATH) if attached else None
print(f"obsm keys: {list(adata.obsm.keys())}")
if attached:
    print(f"X_pca {adata.obsm['X_pca'].shape}  X_state {adata.obsm[SE_KEY].shape}")


In [ ]:
out_path = PATH_TO_DATA / "pbmc_parse_with_splits.h5ad"
adata.write_h5ad(out_path)
print(f"Wrote {out_path}")
print("obsm keys:", list(adata.obsm.keys()))


In [ ]:
# Per-cell Pearson: true log-expression vs PCA reconstruction (all cells, batched)
PCs_T, means = adata.varm["PCs"].T, adata.uns["means_train"]
corrs = []
for start in range(0, adata.n_obs, 50_000):
    sl = slice(start, start + 50_000)
    X_true = adata.X[sl].toarray()
    X_recon = adata.obsm["X_pca"][sl] @ PCs_T + means
    am = X_true - X_true.mean(1, keepdims=True)
    bm = X_recon - X_recon.mean(1, keepdims=True)
    corrs.append((am * bm).sum(1) / np.sqrt((am * am).sum(1) * (bm * bm).sum(1)))
corrs = np.concatenate(corrs)
print(
    f"Per-cell Pearson (true vs recon): mean={corrs.mean():.4f}, "
    f"median={np.median(corrs):.4f}, p5={np.percentile(corrs, 5):.4f}, p95={np.percentile(corrs, 95):.4f}"
)


## Quickly check PCA reconstruction 

In [ ]:
X_pca_reconstructed = adata.obsm["X_pca"] @ adata.varm["PCs"].T 
X_pca_reconstructed = X_pca_reconstructed + adata.uns["means_train"][None, :]

In [ ]:
adata_small = sc.AnnData(X = np.concatenate([adata.X[:10000].toarray(), np.array(X_pca_reconstructed[:10000])], axis=0))

In [ ]:
obs = {"dataset_type": ["true"]*10000 + ["reconstructed"]*10000}
adata_small.obs = pd.DataFrame(obs)

In [ ]:
sc.tl.pca(adata_small)
sc.pp.neighbors(adata_small)
sc.tl.umap(adata_small)
sc.pl.umap(adata_small, color=["dataset_type"], s=10)

## Better understanding in difference across donors 

In [ ]:
sub = adata[adata.obs.donor == "Donor1"].copy()
palette = {c: "lightgray" for c in sub.obs.cytokine.cat.categories}
palette["IL-4"] = "crimson"
palette["PBS"] = "dodgerblue"
sc.pl.umap(
    sub,
    color="cytokine",
    groups=["IL-4", "PBS"],
    palette=palette,
    s=20,
)

In [ ]:
sub = adata[adata.obs.donor == "Donor8"].copy()
palette = {c: "lightgray" for c in sub.obs.cytokine.cat.categories}
palette["IL-4"] = "crimson"
palette["PBS"] = "dodgerblue"
sc.pl.umap(
    sub,
    color="cytokine",
    groups=["IL-4", "PBS"],
    palette=palette,
    s=20,
)

In [ ]:
sub = adata[adata.obs.donor == "Donor2"].copy()
palette = {c: "lightgray" for c in sub.obs.cytokine.cat.categories}
palette["IL-4"] = "crimson"
palette["PBS"] = "dodgerblue"
sc.pl.umap(
    sub,
    color="cytokine",
    groups=["IL-4", "PBS"],
    palette=palette,
    s=20,
)

# Test PCA / SE test set reconstruction 

In [ ]:
adata = sc.read_h5ad(PATH_TO_DATA / "pbmc_parse_with_splits.h5ad")

In [ ]:
adata_train = adata[adata.obs["split3"] == "train"]
adata_finetune = adata[adata.obs["split3"] == "finetune"]
adata_test = adata[adata.obs["split3"] == "test"]

In [ ]:
sc.pp.pca(adata_train)

In [ ]:
PCs_train = adata.varm["PCs"]
means_train = np.asarray(adata.uns["means_train"])

# Finetune recon
z_ft = adata_finetune.X.toarray() - means_train
z_ft = z_ft @ PCs_train
X_recon_ft = z_ft @ PCs_train.T + means_train


# Test set recon 
z_test = adata_test.X.toarray()  - means_train
z_test = z_test @ PCs_train
X_recon_test = z_test @ PCs_train.T + means_train

In [ ]:
adata_test_with_recon = sc.AnnData(X=np.concatenate([adata_test.X.toarray(), X_recon_test], axis=0))
obs_test_with_recon = pd.DataFrame({"dataset_type": ["true"] * adata_test.n_obs + ["recon"] * adata_test.n_obs})
adata_test_with_recon.obs = obs_test_with_recon

In [ ]:
adata_finetune_with_recon = sc.AnnData(X=np.concatenate([adata_finetune.X.toarray(), X_recon_ft], axis=0))
obs_finetune_with_recon = pd.DataFrame({"dataset_type": ["true"] * adata_finetune.n_obs + ["recon"] * adata_finetune.n_obs})
adata_finetune_with_recon.obs = obs_finetune_with_recon

In [ ]:
sc.pp.pca(adata_test_with_recon)
sc.pp.neighbors(adata_test_with_recon)
sc.tl.umap(adata_test_with_recon)
sc.pl.umap(adata_test_with_recon, color=["dataset_type"], s=5)

In [ ]:
sc.pp.pca(adata_finetune_with_recon )
sc.pp.neighbors(adata_finetune_with_recon)
sc.tl.umap(adata_finetune_with_recon)
sc.pl.umap(adata_finetune_with_recon, color=["dataset_type"], s=5)


What if I'd included the ft predictions 

In [ ]:
adata_train_and_ft = adata[adata.obs["split2"].isin(["train", "finetune"])]
sc.tl.pca(adata_train_and_ft)

PCs_train_and_ft = adata_train_and_ft.varm["PCs"]
means_train_and_ft = adata_train_and_ft.X.toarray().mean(0)[None, :]

In [ ]:
# Test set recon 
z_test_train_and_ft = adata_test.X - means_train_and_ft
z_test_train_and_ft = z_test_train_and_ft @ PCs_train_and_ft
X_recon_test_train_and_ft = z_test_train_and_ft @ PCs_train_and_ft.T + means_train_and_ft

In [ ]:
adata_test_with_recon_train_and_ft = sc.AnnData(X=np.concatenate([adata_test.X.toarray(), X_recon_test_train_and_ft], axis=0))
obs_test_with_recon_train_and_ft = pd.DataFrame({"dataset_type": ["true"] * adata_test.n_obs + ["recon"] * adata_test.n_obs})
adata_test_with_recon_train_and_ft.obs = obs_test_with_recon_train_and_ft
sc.pp.pca(adata_test_with_recon_train_and_ft)
sc.pp.neighbors(adata_test_with_recon_train_and_ft)
sc.tl.umap(adata_test_with_recon_train_and_ft)
sc.pl.umap(adata_test_with_recon_train_and_ft, color=["dataset_type"], s=5)